In [1]:
# =============================================================================
# CS-781 Phase 3 — Rhetorical Role prediction (CL + IT) with Arch #1 (decoder-only)
#
# Design
#   * Whole-document context: sentences are packed into one token stream
#         [<s>] s1 </s> s2 </s> s3 </s> ...
#     and the hidden state at each sentence's closing </s> is classified. Because Arch #1 is
#     causal, every sentence sees ALL earlier sentences (up to the window budget).
#   * Long documents are cut into windows: a block of TARGET sentences (labelled) preceded by
#     up to CTX_TOK tokens of earlier sentences as context (not labelled).
#   * Cheap layout features added to the sentence representation (zero-initialised, so they start
#     neutral): relative position of the sentence in the document, absolute sentence index, domain (CL/IT).
#   * Class-weighted loss (sqrt inverse frequency) because the score is macro-F1 over 13 labels.
#   * The same Phase-1 model and tokenizer as Phase 2.
#
# Techniques (TECHNIQUES): full | lorma | lora | adapter | partial | normtune | kd
#
# Context ablation (needed for the report: "does context help?"): set
#     TRAIN_TARGET_TOKS=[1]; INFER_TARGET_TOK=1; CTX_TOK=0; USE_POSFEAT=False
# -> every sentence is classified alone (sentence-only baseline).
#
# Kaggle: paste each "# %%" block as its own cell, in order. Check the 3 paths in cell [2].
# =============================================================================


# %% [1] Imports & HF login
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json, math, random, time, zipfile, re, pickle, gc, traceback, functools
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_recall_fscore_support

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# %% [2] Config
# ---- RUN MODE (edit this one line, then Save & Run All) -------------------------------------
#   "kd"           : distillation run (teacher = your saved full model, see TEACHER_CKPT below)
#   "final_all"    : retrain full/adapter/lorma on train+dev (no dev score) -> *_all outputs
#   "ensemble_only": no training, only the final ensemble cell [12] on already-saved probabilities
MODE = "kd"   # also: "ctx_ablation" = sentence-only baseline (each sentence classified alone) for the "does context help?" analysis
TEACHER_CKPT = ""   # path to your Phase-3 full rr_best.pt (e.g. /kaggle/input/YOUR_DS/rr_best.pt). "" = look in working dir
TECHNIQUES = {"kd": ["kd"], "final_all": ["full", "adapter", "lorma"], "ensemble_only": [], "ctx_ablation": ["full"]}[MODE]
SEED = 42
SKIP_DONE = True
FINAL_ALL = (MODE == "final_all")   # True = train on train+dev with the fixed epochs in HP

ARTIFACT_PATH = "/kaggle/input/datasets/akashmalgav2/arch1final"                        # tokenizer (legal_bpe.json)
MODEL_CKPT_PATH = "/kaggle/input/datasets/akashmalgav2/phase2-model1/model_best.pt"        # Arch #1 Phase-1 checkpoint

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
FILES = {
    "CL_train": "Phase_3_RR/Train/CL_train.json", "CL_dev": "Phase_3_RR/Train/CL_dev.json",
    "IT_train": "Phase_3_RR/Train/IT_train.json", "IT_dev": "Phase_3_RR/Train/IT_dev.json",
    "CL_test": "Phase_3_RR/Test/CL_test.json",    "IT_test": "Phase_3_RR/Test/IT_test.json",
}
VOCAB_FILE = "Phase_3_RR/Train/label_vocab.json"

MAX_SENT_TOKENS = 128         # longer sentences are cut to their first 128 tokens
TRAIN_TARGET_TOKS = [640, 400]  # two window layouts per document (acts as augmentation)
INFER_TARGET_TOK = 640
CTX_TOK = 320                 # tokens of earlier sentences given as context to every window
USE_POSFEAT = True            # sentence-position embeddings
USE_DOMAIN = True             # CL / IT embedding
if MODE == "ctx_ablation":    # sentence-only baseline: one sentence per window, no earlier context, no position features
    TRAIN_TARGET_TOKS = [1]; INFER_TARGET_TOK = 1; CTX_TOK = 0; USE_POSFEAT = False
CLASS_WEIGHT = "sqrt"         # "none" | "sqrt" | "inv"

BATCH_SIZE = 8
EVAL_BATCH = 8
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
DROPOUT_FT = 0.1

LORMA_R, LORMA_ALPHA = 8, 8
ADAPTER_DIM = 32
PARTIAL_LAYERS = 4
STUDENT_KEEP = [0, 1, 3, 4, 6, 7, 9, 10, 11]   # KD student: 9 of 12 Phase-1 layers (a guess; the 6-layer student was too damaged in Phase 2)
KD_T, KD_ALPHA = 2.0, 0.5

HP = {   # starting points, not tuned
    "full":     dict(epochs=8,  lr_body=5e-5, lr_head=5e-4),
    "lorma":    dict(epochs=15, lr_body=4e-4, lr_head=1e-3),
    "lora":     dict(epochs=15, lr_body=4e-4, lr_head=1e-3),
    "adapter":  dict(epochs=15, lr_body=4e-4, lr_head=1e-3),
    "partial":  dict(epochs=10, lr_body=1e-4, lr_head=5e-4),
    "normtune": dict(epochs=20, lr_body=1e-3, lr_head=1e-3),
    "kd":       dict(epochs=10, lr_body=1e-4, lr_head=5e-4),
}
for _t in TECHNIQUES:
    assert _t in HP, _t
if "kd" in TECHNIQUES and "full" in TECHNIQUES:
    assert TECHNIQUES.index("full") < TECHNIQUES.index("kd"), "put 'full' before 'kd'"

ROOT_OUT = "/kaggle/working"
def out_dir_for(tech):
    return f"{ROOT_OUT}/cs781_phase3_arch1_{tech}_s{SEED}" + ("_all" if FINAL_ALL else "") + ("_sentonly" if MODE == "ctx_ablation" else "")
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)
print("Will run:", TECHNIQUES, "| seed:", SEED, "| final-all:", FINAL_ALL)
# %% [3] Tokenizer (Phase-1 BPE) with a cached sentence encoder
with open(f"{ARTIFACT_PATH}/legal_bpe.json", "r", encoding="utf-8") as f:
    _tok = json.load(f)
vocab = _tok["itos"]
merges = [tuple(p) for p in _tok["merges"]]
token_to_id = {tok: i for i, tok in enumerate(vocab)}
merges_rank = {p: i for i, p in enumerate(merges)}
PAD_ID, UNK_ID = token_to_id["<pad>"], token_to_id["<unk>"]
BOS_ID, EOS_ID = token_to_id["<s>"], token_to_id["</s>"]
SEP_ID = EOS_ID                       # </s> closes every sentence; its hidden state represents the sentence
VOCAB_SIZE = len(vocab)
EOW = "\u0100"

def word_to_symbols(word):
    return tuple(chr(b) for b in word.encode("utf-8", "ignore")) + (EOW,)

def bpe_encode_word(word):
    symbols = list(word_to_symbols(word))
    while len(symbols) > 1:
        pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
        ranked = [(merges_rank[p], p) for p in pairs if p in merges_rank]
        if not ranked:
            break
        _, best = min(ranked)
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == best[0] and symbols[i + 1] == best[1]:
                new_symbols.append(best[0] + best[1]); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        symbols = new_symbols
    return symbols

@functools.lru_cache(maxsize=None)
def encode_word_cached(word):
    return tuple(token_to_id.get(sym, UNK_ID) for sym in bpe_encode_word(word))

def encode_sentence(text):
    ids = []
    for w in str(text).split():
        ids.extend(encode_word_cached(w))
        if len(ids) >= MAX_SENT_TOKENS:
            break
    return ids[:MAX_SENT_TOKENS]


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cuda
Will run: ['kd'] | seed: 42 | final-all: False


In [2]:

# %% [4] Load Phase-3 data

def hf_local(path):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=path,
        repo_type="dataset",
        token=HF_TOKEN
    )


def parse_vocab(raw):
    if isinstance(raw, list):
        labs = list(raw)

    elif isinstance(raw, dict) and raw:
        vals = list(raw.values())

        if all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            # label -> id
            labs = [
                k for k, _ in sorted(
                    raw.items(), key=lambda kv: kv[1]
                )
            ]
        else:
            # id -> label
            try:
                labs = [
                    v for _, v in sorted(
                        raw.items(), key=lambda kv: int(kv[0])
                    )
                ]
            except ValueError:
                labs = vals

    else:
        raise ValueError("Unsupported label_vocab.json format")

    labs = ["None" if l is None else str(l) for l in labs]

    assert len(labs) == 13 and len(set(labs)) == 13, labs

    return labs


LABELS = parse_vocab(
    json.load(open(hf_local(VOCAB_FILE), encoding="utf-8"))
)
LAB2ID = {l: i for i, l in enumerate(LABELS)}
NUM_CLS = len(LABELS)

print("Labels:", LABELS)

N_REL = 20
ABS_EDGES = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 512])
N_ABS = len(ABS_EDGES) + 1


def load_docs(path, domain):
    raw = json.load(open(hf_local(path), encoding="utf-8"))

    items = []

    if isinstance(raw, list):
        for i, d in enumerate(raw):
            if not isinstance(d, dict):
                raise ValueError(
                    f"{path}: record at index {i} is {type(d)}, "
                    "expected a dictionary"
                )
            items.append((str(d.get("id", i)), d))

    elif isinstance(raw, dict):
        for k, v in raw.items():

            # Some test files store the document's sentences
            # directly as a list under its document ID.
            if isinstance(v, list):
                d = {
                    "id": str(k),
                    "text": v
                }

            elif isinstance(v, dict):
                d = dict(v)
                d.setdefault("id", str(k))

            else:
                raise ValueError(
                    f"{path}: unexpected record type {type(v)} "
                    f"for document ID {k}"
                )

            items.append((str(d["id"]), d))

    else:
        raise ValueError(
            f"{path}: unexpected top-level type {type(raw)}"
        )

    docs = []

    for did, d in items:
        if "text" not in d:
            raise ValueError(
                f"{path}: document {did} has no 'text' field. "
                f"Available keys: {list(d.keys())}"
            )

        sents = d["text"]

        if isinstance(sents, str):
            raise ValueError(
                f"{path}: 'text' is a string, expected a list "
                f"of sentences (document {did})"
            )

        if not isinstance(sents, list):
            raise ValueError(
                f"{path}: 'text' must be a list for document {did}; "
                f"got {type(sents)}"
            )

        labs = d.get("labels")

        if labs is not None:
            assert len(labs) == len(sents), (
                f"{path} doc {did}: {len(labs)} labels vs "
                f"{len(sents)} sentences"
            )

            labs = [
                LAB2ID["None" if l is None else str(l)]
                for l in labs
            ]

        sent_ids = [encode_sentence(s) + [SEP_ID] for s in sents]
        n = len(sent_ids)

        if n == 0:
            continue

        prel = [
            min(int(N_REL * i / n), N_REL - 1)
            for i in range(n)
        ]

        pabs = np.searchsorted(
            ABS_EDGES, np.arange(n), side="right"
        ).tolist()

        docs.append({
            "id": did,
            "domain": domain,
            "sent_ids": sent_ids,
            "lens": [len(s) for s in sent_ids],
            "labels": labs,
            "n": n,
            "prel": prel,
            "pabs": pabs
        })

    return docs


t0 = time.time()

train_recs = (
    load_docs(FILES["CL_train"], 0)
    + load_docs(FILES["IT_train"], 1)
)

dev_recs = (
    load_docs(FILES["CL_dev"], 0)
    + load_docs(FILES["IT_dev"], 1)
)

test_recs = (
    load_docs(FILES["CL_test"], 0)
    + load_docs(FILES["IT_test"], 1)
)

print(
    f"Loaded in {time.time() - t0:.0f}s | "
    f"docs: train {len(train_recs)} "
    f"dev {len(dev_recs)} test {len(test_recs)}"
)

for name, rs in [
    ("train", train_recs),
    ("dev", dev_recs),
    ("test", test_recs)
]:
    for dom, dn in [(0, "CL"), (1, "IT")]:
        sub = [r for r in rs if r["domain"] == dom]

        if sub:
            ns = [r["n"] for r in sub]

            print(
                f"  {name} {dn}: {len(sub)} docs | "
                f"sentences {sum(ns)} | "
                f"per doc median {int(np.median(ns))} "
                f"max {max(ns)}"
            )

tok_per_sent = np.mean([
    l for r in train_recs for l in r["lens"]
])

print(
    f"Average tokens per sentence (incl. </s>): "
    f"{tok_per_sent:.1f}"
)

train_counts = Counter(
    l for r in train_recs for l in r["labels"]
)

print(
    "Train label counts:",
    {
        LABELS[i]: train_counts.get(i, 0)
        for i in range(NUM_CLS)
    }
)

all_ids = [r["id"] for r in test_recs]

assert len(all_ids) == len(set(all_ids)), (
    "test doc IDs collide across CL / IT"
)
# %% [5] Windows and datasets
def make_sent_windows(lens, target_tok, ctx_tok):
    """Windows (c, a, b): sentences [a, b) are the labelled targets, [c, a) are unlabelled earlier context."""
    wins, i, n = [], 0, len(lens)
    while i < n:
        j, tot = i, 0
        while j < n and (j == i or tot + lens[j] <= target_tok):
            tot += lens[j]; j += 1
        c, ctot = i, 0
        while c > 0 and ctot + lens[c - 1] <= ctx_tok:
            c -= 1; ctot += lens[c]
        wins.append((c, i, j))
        i = j
    return wins

def build_window(rec, c, a, b, with_labels):
    ids, sep_pos, pos = [BOS_ID], [], 1
    for i in range(c, b):
        ids.extend(rec["sent_ids"][i])
        pos += rec["lens"][i]
        if i >= a:
            sep_pos.append(pos - 1)               # index of this sentence's closing </s>
    return {"ids": ids, "sep_pos": sep_pos, "domain": rec["domain"],
            "labels": rec["labels"][a:b] if with_labels else None,
            "prel": rec["prel"][a:b], "pabs": rec["pabs"][a:b]}

def collate(batch):
    B = len(batch)
    T = max(len(x["ids"]) for x in batch)
    S = max(len(x["sep_pos"]) for x in batch)
    ids = torch.full((B, T), PAD_ID, dtype=torch.long)
    sep = torch.zeros((B, S), dtype=torch.long)
    lab = torch.full((B, S), -100, dtype=torch.long)
    prel = torch.zeros((B, S), dtype=torch.long)
    pabs = torch.zeros((B, S), dtype=torch.long)
    dom = torch.zeros((B,), dtype=torch.long)
    for i, x in enumerate(batch):
        L, s = len(x["ids"]), len(x["sep_pos"])
        ids[i, :L] = torch.tensor(x["ids"])
        sep[i, :s] = torch.tensor(x["sep_pos"])
        prel[i, :s] = torch.tensor(x["prel"])
        pabs[i, :s] = torch.tensor(x["pabs"])
        dom[i] = x["domain"]
        if x["labels"] is not None:
            lab[i, :s] = torch.tensor(x["labels"])
    return ids, sep, lab, prel, pabs, dom

class WindowDataset(Dataset):
    def __init__(self, recs_list):
        self.items = []
        for r in recs_list:
            for tt in TRAIN_TARGET_TOKS:
                for (c, a, b) in make_sent_windows(r["lens"], tt, CTX_TOK):
                    self.items.append(build_window(r, c, a, b, True))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

fit_recs = train_recs + dev_recs if FINAL_ALL else train_recs
eval_recs = [] if FINAL_ALL else dev_recs
train_ds = WindowDataset(fit_recs)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=0)
max_len = max(len(x["ids"]) for x in train_ds.items)
print(f"Training windows: {len(train_ds)} | steps/epoch: {len(train_loader)} | longest window {max_len} tokens")

# class weights from the training labels
cnt = np.array([sum(1 for r in fit_recs for l in r["labels"] if l == k) for k in range(NUM_CLS)], dtype=np.float64)
if CLASS_WEIGHT == "none":
    cw = np.ones(NUM_CLS)
else:
    power = 0.5 if CLASS_WEIGHT == "sqrt" else 1.0
    cw = np.where(cnt > 0, (cnt.sum() / (NUM_CLS * np.maximum(cnt, 1))) ** power, 1.0)
    cw = cw / cw.mean()
class_w = torch.tensor(cw, dtype=torch.float32).to(device)
print("Class weights:", {LABELS[i]: round(float(cw[i]), 2) for i in range(NUM_CLS)})
# %% [6] Model: Arch #1 backbone + PEFT modules + sentence head
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def build_rope_cache(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class LoRMAPlus(nn.Module):
    """h' = h + (alpha/r) B(A h) = ((alpha/r) B A + I) W0 x with h = W0 x (multiplicative; B = 0 at init)."""
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, h):
        return h + self.scale * F.linear(F.linear(h, self.A), self.B)

class LoRAAdapter(nn.Module):
    """Additive update (alpha/r) B(A x), added to W0 x (LoRA; B = 0 at init)."""
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, x):
        return self.scale * F.linear(F.linear(x, self.A), self.B)

class BottleneckAdapter(nn.Module):
    """Houlsby-style adapter: h + up(gelu(down(h))); `up` starts at zero (identity at init)."""
    def __init__(self, d, m):
        super().__init__()
        self.down = nn.Linear(d, m)
        self.up = nn.Linear(m, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.n_head, self.head_dim, self.n_embd = n_head, n_embd // n_head, n_embd
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
        self.adapt_mode = None
        self.q_ad = None
        self.v_ad = None
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        if self.adapt_mode == "lorma":
            q, v = self.q_ad(q), self.v_ad(v)
        elif self.adapt_mode == "lora":
            q, v = q + self.q_ad(x), v + self.v_ad(x)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos[:, :, :T, :], sin[:, :, :T, :])
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.proj(out.transpose(1, 2).contiguous().view(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, mult, dropout):
        super().__init__()
        hidden = ((int(mult * n_embd) + 63) // 64) * 64
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(hidden, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, ffn_mult, dropout):
        super().__init__()
        self.ln1 = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = RMSNorm(n_embd)
        self.ffn = SwiGLU(n_embd, ffn_mult, dropout)
        self.ad1 = None
        self.ad2 = None
    def forward(self, x, cos, sin):
        a = self.attn(self.ln1(x), cos, sin)
        if self.ad1 is not None:
            a = self.ad1(a)
        x = x + a
        f = self.ffn(self.ln2(x))
        if self.ad2 is not None:
            f = self.ad2(f)
        return x + f

class LegalLLaMA(nn.Module):
    def __init__(self, vocab_size, n_layer, n_head, n_embd, block_size, ffn_mult, dropout):
        super().__init__()
        self.block_size, self.head_dim = block_size, n_embd // n_head
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, ffn_mult, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight
        cos, sin = build_rope_cache(block_size, self.head_dim, device="cpu")
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def forward_hidden(self, idx):
        x = self.drop(self.tok_emb(idx))
        cos, sin = self.rope_cos.to(idx.device), self.rope_sin.to(idx.device)
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return self.ln_f(x)

class SentModel(nn.Module):
    """Classifies the hidden state at each sentence's closing </s>."""
    def __init__(self, backbone, d, n_cls, dropout, use_pos, use_dom):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(d, n_cls)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
        self.use_pos, self.use_dom = use_pos, use_dom
        if use_pos:
            self.rel_emb = nn.Embedding(N_REL, d)
            self.abs_emb = nn.Embedding(N_ABS, d)
            nn.init.zeros_(self.rel_emb.weight); nn.init.zeros_(self.abs_emb.weight)
        if use_dom:
            self.dom_emb = nn.Embedding(2, d)
            nn.init.zeros_(self.dom_emb.weight)
    def head_parameters(self):
        ps = list(self.classifier.parameters())
        if self.use_pos:
            ps += list(self.rel_emb.parameters()) + list(self.abs_emb.parameters())
        if self.use_dom:
            ps += list(self.dom_emb.parameters())
        return ps
    def forward(self, ids, sep_pos, prel, pabs, dom):
        h = self.backbone.forward_hidden(ids)                                   # (B, T, d)
        hs = torch.gather(h, 1, sep_pos.unsqueeze(-1).expand(-1, -1, h.size(-1)))  # (B, S, d)
        if self.use_pos:
            hs = hs + self.rel_emb(prel) + self.abs_emb(pabs)
        if self.use_dom:
            hs = hs + self.dom_emb(dom).unsqueeze(1)
        return self.classifier(self.drop(hs))                                   # (B, S, n_cls)

ckpt = torch.load(MODEL_CKPT_PATH, map_location="cpu")
cfg = ckpt["config"]
assert cfg["vocab_size"] == VOCAB_SIZE, (cfg["vocab_size"], VOCAB_SIZE)
assert max_len <= cfg["block_size"], f"window of {max_len} tokens exceeds the model context {cfg['block_size']}: lower the *_TOK settings"
print(f"Phase-1 model: {cfg['n_layer']} layers, d={cfg['n_embd']}, context {cfg['block_size']}")

def make_student_state(sd, keep):
    new = {}
    for k, v in sd.items():
        if k.startswith("blocks."):
            parts = k.split(".")
            old = int(parts[1])
            if old in keep:
                parts[1] = str(keep.index(old))
                new[".".join(parts)] = v
        else:
            new[k] = v
    return new

def build_model(technique, keep_layers=None):
    sd, n_layer = ckpt["model_state_dict"], cfg["n_layer"]
    if keep_layers is not None:
        sd, n_layer = make_student_state(sd, keep_layers), len(keep_layers)
    bb = LegalLLaMA(cfg["vocab_size"], n_layer, cfg["n_head"], cfg["n_embd"],
                    cfg["block_size"], cfg["ffn_mult"], DROPOUT_FT)
    bb.load_state_dict(sd)
    d = cfg["n_embd"]
    if technique in ("lorma", "lora", "adapter", "partial", "normtune"):
        for p in bb.parameters():
            p.requires_grad = False
        if technique in ("lorma", "lora"):
            for blk in bb.blocks:
                blk.attn.adapt_mode = technique
                make = LoRMAPlus if technique == "lorma" else LoRAAdapter
                blk.attn.q_ad, blk.attn.v_ad = make(d, LORMA_R, LORMA_ALPHA), make(d, LORMA_R, LORMA_ALPHA)
        elif technique == "adapter":
            for blk in bb.blocks:
                blk.ad1, blk.ad2 = BottleneckAdapter(d, ADAPTER_DIM), BottleneckAdapter(d, ADAPTER_DIM)
        elif technique == "partial":
            for blk in bb.blocks[-PARTIAL_LAYERS:]:
                for p in blk.parameters():
                    p.requires_grad = True
            for p in bb.ln_f.parameters():
                p.requires_grad = True
        elif technique == "normtune":
            for m in bb.modules():
                if isinstance(m, RMSNorm):
                    m.weight.requires_grad = True
    net = SentModel(bb, d, NUM_CLS, DROPOUT_FT, USE_POSFEAT, USE_DOMAIN).to(device)
    n_tot = sum(p.numel() for p in net.parameters())
    n_trn = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print(f"[{technique}{' student' if keep_layers else ''}] trainable {n_trn/1e6:.3f}M of {n_tot/1e6:.1f}M params ({100*n_trn/n_tot:.2f}%)")
    return net

# %% [7] Inference and metrics (macro-F1 pooled over CL + IT, as in the competition scorer)
@torch.no_grad()
def predict_probs(net, recs_list):
    net.eval()
    out = {r["id"]: np.zeros((r["n"], NUM_CLS), dtype=np.float32) for r in recs_list}
    items = []
    for r in recs_list:
        for (c, a, b) in make_sent_windows(r["lens"], INFER_TARGET_TOK, CTX_TOK):
            items.append((r["id"], a, b, build_window(r, c, a, b, False)))
    for i in range(0, len(items), EVAL_BATCH):
        chunk = items[i:i + EVAL_BATCH]
        ids, sep, _, prel, pabs, dom = collate([x[3] for x in chunk])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = net(ids.to(device), sep.to(device), prel.to(device), pabs.to(device), dom.to(device))
        probs = F.softmax(logits.float(), dim=-1).cpu().numpy()
        for j, (rid, a, b, _) in enumerate(chunk):
            out[rid][a:b] = probs[j, :b - a]
    return out

def score(probs_by_id, recs_list):
    y_true = [l for r in recs_list for l in r["labels"]]
    y_pred = [int(k) for r in recs_list for k in probs_by_id[r["id"]].argmax(1)]
    labs = list(range(NUM_CLS))
    f1 = f1_score(y_true, y_pred, labels=labs, average="macro", zero_division=0)
    p, r_, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labs, zero_division=0)
    return f1, float(np.mean(p)), float(np.mean(r_)), (p, r_, f, s), y_true, y_pred

def report(probs_by_id, recs_list):
    f1, mp, mr, (p, r_, f, s), y_true, y_pred = score(probs_by_id, recs_list)
    print(f"dev pooled macro-F1 {f1:.4f} | macro-P {mp:.4f} | macro-R {mr:.4f}")
    for dom, dn in [(0, "CL"), (1, "IT")]:
        sub = [r for r in recs_list if r["domain"] == dom]
        if sub:
            print(f"  {dn} alone: macro-F1 {score(probs_by_id, sub)[0]:.4f}  ({len(sub)} docs)")
    print(f"  {'label':24s} {'P':>6s} {'R':>6s} {'F1':>6s} {'support':>8s}")
    for k in range(NUM_CLS):
        print(f"  {LABELS[k]:24s} {p[k]:6.3f} {r_[k]:6.3f} {f[k]:6.3f} {int(s[k]):8d}")
    conf = Counter((a, b) for a, b in zip(y_true, y_pred) if a != b)
    print("  most frequent confusions (true -> predicted):")
    for (a, b), c in conf.most_common(10):
        print(f"    {LABELS[a]} -> {LABELS[b]}: {c}")
    return f1

# %% [8] Training function (optionally with a teacher for distillation)
def fit(model, epochs, lr_body, lr_head, best_path, teacher=None):
    head_params = model.head_parameters()
    head_ids = {id(p) for p in head_params}
    body_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    trainable = [p for p in model.parameters() if p.requires_grad]
    groups = [{"params": head_params, "lr": lr_head, "base": lr_head}]
    if body_params:
        groups.insert(0, {"params": body_params, "lr": lr_body, "base": lr_body})
    optimizer = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    total_steps = epochs * len(train_loader)
    warm = max(1, int(WARMUP_FRAC * total_steps))
    def lr_factor(step):
        if step < warm:
            return step / warm
        prog = (step - warm) / max(1, total_steps - warm)
        return max(0.02, 0.5 * (1 + math.cos(math.pi * prog)))

    best_f1, step, t0, history = -1.0, 0, time.time(), []
    if teacher is not None:
        teacher.eval()
    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, n_b = 0.0, 0
        for ids, sep, lab, prel, pabs, dom in train_loader:
            ids, sep, lab = ids.to(device), sep.to(device), lab.to(device)
            prel, pabs, dom = prel.to(device), pabs.to(device), dom.to(device)
            for g in optimizer.param_groups:
                g["lr"] = g["base"] * lr_factor(step)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                logits = model(ids, sep, prel, pabs, dom)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits = teacher(ids, sep, prel, pabs, dom)
            logits = logits.float()
            loss = F.cross_entropy(logits.view(-1, NUM_CLS), lab.view(-1), weight=class_w, ignore_index=-100)
            if teacher is not None:
                mask = lab != -100
                if mask.any():
                    s_ = F.log_softmax(logits[mask] / KD_T, dim=-1)
                    t_ = F.softmax(t_logits.float()[mask] / KD_T, dim=-1)
                    kd = F.kl_div(s_, t_, reduction="batchmean") * (KD_T ** 2)
                    loss = (1 - KD_ALPHA) * loss + KD_ALPHA * kd
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item(); n_b += 1; step += 1
        rec_h = {"epoch": epoch, "train_loss": run_loss / max(1, n_b)}
        if eval_recs:
            f1, mp, mr, *_ = score(predict_probs(model, eval_recs), eval_recs)
            rec_h["dev_f1"] = f1
            print(f"[epoch {epoch}] loss {rec_h['train_loss']:.4f} | dev macro-F1 {f1:.4f} (P {mp:.4f} R {mr:.4f}) | {(time.time()-t0)/60:.1f}m")
            if f1 > best_f1:
                best_f1 = f1
                torch.save({"model": model.state_dict(), "epoch": epoch, "dev_f1": f1}, best_path)
                print("  -> new best checkpoint saved")
        else:
            print(f"[epoch {epoch}] loss {rec_h['train_loss']:.4f} (final run on train+dev; no dev score) | {(time.time()-t0)/60:.1f}m")
            torch.save({"model": model.state_dict(), "epoch": epoch, "dev_f1": None}, best_path)
        history.append(rec_h)
    model.load_state_dict(torch.load(best_path, map_location="cpu")["model"])
    return history

# %% [9] One technique end-to-end (train -> dev report -> test predictions -> save probabilities)
def set_seed(s):
    random.seed(s); torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def run_technique(tech):
    out_dir = out_dir_for(tech)
    os.makedirs(out_dir, exist_ok=True)
    best_path = os.path.join(out_dir, "rr_best.pt")
    pred_path = os.path.join(out_dir, "prediction.json")
    summ_path = os.path.join(out_dir, "summary.json")
    prob_path = os.path.join(out_dir, "sentence_probs.pkl")
    if SKIP_DONE and all(os.path.exists(p) for p in [pred_path, summ_path, prob_path]):
        print(f"\n##### [{tech}] already finished -> skipping")
        return json.load(open(summ_path))

    print(f"\n{'='*78}\n##### Phase 3 | Arch1 | TECHNIQUE: {tech} | seed {SEED} | {HP[tech]}\n{'='*78}")
    set_seed(SEED)
    t_run = time.time()
    teacher = None
    if tech == "kd":
        teacher = build_model("full")
        full_ckpt = TEACHER_CKPT or os.path.join(out_dir_for("full"), "rr_best.pt")
        if os.path.exists(full_ckpt):
            teacher.load_state_dict(torch.load(full_ckpt, map_location="cpu")["model"])
            print("Loaded teacher from", full_ckpt)
        else:
            print("No saved teacher found -> training the teacher (full fine-tuning) first")
            fit(teacher, best_path=os.path.join(out_dir, "teacher_best.pt"), **HP["full"])
        for p in teacher.parameters():
            p.requires_grad = False
        teacher.eval()
        if eval_recs:
            print(f"TEACHER dev macro-F1: {score(predict_probs(teacher, eval_recs), eval_recs)[0]:.4f}")
        model = build_model("full", keep_layers=STUDENT_KEEP)
        t_p = sum(p.numel() for p in teacher.parameters())
        s_p = sum(p.numel() for p in model.parameters())
        print(f"Teacher {t_p/1e6:.1f}M -> student {s_p/1e6:.1f}M params ({100*(1-s_p/t_p):.0f}% smaller)")
        history = fit(model, best_path=best_path, teacher=teacher, **HP["kd"])
    else:
        model = build_model(tech)
        history = fit(model, best_path=best_path, **HP[tech])
    json.dump(history, open(os.path.join(out_dir, "history.json"), "w"), indent=2)
    print(f"[{tech}] training finished in {(time.time()-t_run)/60:.1f} min")

    dev_f1, dev_probs = None, {}
    if eval_recs:
        dev_probs = predict_probs(model, eval_recs)
        dev_f1 = report(dev_probs, eval_recs)

    test_probs = predict_probs(model, test_recs)
    pred = {}
    for r in test_recs:
        labs = [LABELS[int(k)] for k in test_probs[r["id"]].argmax(1)]
        assert len(labs) == r["n"]
        pred[r["id"]] = labs
    assert set(pred) == {r["id"] for r in test_recs}
    assert all(l in LAB2ID for v in pred.values() for l in v)
    cnt_pred = Counter(l for v in pred.values() for l in v)
    print(f"[{tech}] test docs {len(pred)} | predicted label counts:", dict(cnt_pred.most_common()))

    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(pred, f)
    pickle.dump({"labels": LABELS, "dev": {k: v.astype(np.float16) for k, v in dev_probs.items()},
                 "test": {k: v.astype(np.float16) for k, v in test_probs.items()}}, open(prob_path, "wb"))
    summary = {"technique": tech, "dev_f1": dev_f1, "minutes": round((time.time() - t_run) / 60, 1)}
    json.dump(summary, open(summ_path, "w"), indent=2)
    del model, teacher
    return summary


# %% [10] Run all chosen techniques
all_results = []
for tech in TECHNIQUES:
    try:
        res = run_technique(tech)
    except Exception:
        print(f"\n!!!!! [{tech}] FAILED -- continuing with the next technique")
        traceback.print_exc()
        res = {"technique": tech, "error": True}
    all_results.append(res)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 78 + "\nSUMMARY (dev macro-F1, CL + IT pooled)\n" + "=" * 78)
for r in all_results:
    if r.get("error"):
        print(f"{r['technique']:10s}  FAILED")
    else:
        d = f"{r['dev_f1']:.4f}" if r["dev_f1"] is not None else "n/a"
        print(f"{r['technique']:10s} dev macro-F1 {d} | {r['minutes']} min")
# %% [11] Zips: one Codabench submission zip per technique + one results zip (probabilities for ensembling)
SUB_DIR = f"{ROOT_OUT}/submissions"
RES_DIR = f"{ROOT_OUT}/results_zips"
os.makedirs(SUB_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)
for t in TECHNIQUES:
    d = out_dir_for(t)
    pred = f"{d}/prediction.json"
    if not os.path.exists(pred):
        print(f"[{t}] prediction.json not found -> skipped")
        continue
    tag = f"phase3_arch1_{t}_s{SEED}" + ("_all" if FINAL_ALL else "") + ("_sentonly" if MODE == "ctx_ablation" else "")
    zpath = f"{SUB_DIR}/submission_{tag}.zip"
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred, arcname="prediction.json")          # prediction.json at the zip root
    rpath = f"{RES_DIR}/results_{tag}.zip"
    with zipfile.ZipFile(rpath, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fn in ["prediction.json", "sentence_probs.pkl", "history.json", "summary.json"]:
            if os.path.exists(f"{d}/{fn}"):
                zf.write(f"{d}/{fn}", arcname=fn)
    print(f"[{t}] submission: {os.path.basename(zpath)} | results: {os.path.basename(rpath)}")
print("Upload the zips in", SUB_DIR, "to Codabench.")

# %% [12] Final ensemble + Viterbi (no GPU). Runs in modes "final_all" and "ensemble_only".
# Weights/lambda come from the dev search you already did (full 2, lorma 1, adapter 2, lambda 0.25).
if MODE in ("final_all", "ensemble_only"):
    ENS_LAMBDA = 0.25
    W = f"{ROOT_OUT}/cs781_phase3_arch1_"
    ENS_SOURCES = {   # name: (sentence_probs.pkl path, weight)  -- add/remove freely; missing files are skipped
        "full_all":    (W + "full_s42_all/sentence_probs.pkl",    2),
        "lorma_all":   (W + "lorma_s42_all/sentence_probs.pkl",   1),
        "adapter_all": (W + "adapter_s42_all/sentence_probs.pkl", 2),
        # earlier dev-trained runs can be mixed in too, e.g.:
        # "full_dev":  ("/kaggle/input/YOUR_DS/sentence_probs_full.pkl", 1),
    }
    src = {}
    for name, (p, w) in ENS_SOURCES.items():
        if os.path.exists(p):
            d = pickle.load(open(p, "rb")); assert d["labels"] == LABELS, name
            src[name] = (d["test"], w)
        else:
            print("skip (not found):", p)
    assert src, "no probability files found"
    print("ensemble members:", {k: v[1] for k, v in src.items()})

    K = NUM_CLS
    _recs_for_trans = train_recs + dev_recs
    trans = np.full((K, K), 0.5); start = np.full(K, 0.5); prior = np.full(K, 0.5)
    for r in _recs_for_trans:
        l = r["labels"]; start[l[0]] += 1
        for a in l: prior[a] += 1
        for a, b in zip(l[:-1], l[1:]): trans[a, b] += 1
    prior_p = prior / prior.sum()
    pmi = np.log(trans / trans.sum(1, keepdims=True)) - np.log(prior_p)[None, :]
    start_pmi = np.log(start / start.sum()) - np.log(prior_p)

    def decode_seq(P, lam):
        lp = np.log(np.clip(P, 1e-8, 1.0)); n = len(lp)
        if lam == 0 or n == 1:
            return lp.argmax(1)
        score = lp[0] + lam * start_pmi
        back = np.zeros((n, K), dtype=np.int64)
        for t in range(1, n):
            cand = score[:, None] + lam * pmi
            back[t] = cand.argmax(0); score = cand.max(0) + lp[t]
        path = np.zeros(n, dtype=np.int64); path[-1] = score.argmax()
        for t in range(n - 1, 0, -1):
            path[t - 1] = back[t, path[t]]
        return path

    tot_w = sum(w for _, w in src.values())
    pred = {}
    for r in test_recs:
        P = sum(w * d[r["id"]].astype(np.float32) for d, w in src.values()) / tot_w
        labs = [LABELS[int(k)] for k in decode_seq(P, ENS_LAMBDA)]
        assert len(labs) == r["n"]
        pred[r["id"]] = labs
    assert set(pred) == {r["id"] for r in test_recs} and all(l in LAB2ID for v in pred.values() for l in v)
    print("predicted label counts:", dict(Counter(l for v in pred.values() for l in v).most_common()))
    ENS_DIR = f"{ROOT_OUT}/phase3_final_ensemble"; os.makedirs(ENS_DIR, exist_ok=True)
    json.dump(pred, open(f"{ENS_DIR}/prediction.json", "w"))
    zp = f"{ENS_DIR}/submission_phase3_ens_{MODE}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(f"{ENS_DIR}/prediction.json", arcname="prediction.json")
    print("wrote", zp)

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

Loaded in 2s | docs: train 80 dev 10 test 10
  train CL: 40 docs | sentences 10593 | per doc median 120 max 1816
  train IT: 40 docs | sentences 6252 | per doc median 126 max 442
  dev CL: 5 docs | sentences 1396 | per doc median 184 max 848
  dev IT: 5 docs | sentences 746 | per doc median 120 max 382
  test CL: 5 docs | sentences 1339 | per doc median 114 max 942
  test IT: 5 docs | sentences 858 | per doc median 159 max 420
Average tokens per sentence (incl. </s>): 37.5
Train label counts: {'Fact': 4098, 'Issue': 107, 'ArgumentPetitioner': 1613, 'ArgumentRespondent': 1348, 'Statute': 1099, 'Dissent': 284, 'PrecedentReliedUpon': 1945, 'PrecedentNotReliedUpon': 569, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 874, 'RatioOfTheDecision': 4301, 'RulingByPresentCourt': 307, 'None': 297}
Training windows: 2793 | steps/epoch: 350 | longest window 960 tokens
Class weights: {'Fact': 0.19, 'Issue': 1.19, 'ArgumentPetitioner': 0.31, 'ArgumentRespondent': 0.33, 'Statute': 0.37, 'Dissent': 0.7